In [1]:
from incidentiq.pipeline import IncidentIQ
from incidentiq.reasoning.models import InvestigationState

e:\incidentiq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "../data/processed/logs.parquet"
MODEL = "gemini-3.5-flash-lite"

incidentiq = IncidentIQ(
    data_path=DATA_PATH,
    model=MODEL,
)

print("IncidentIQ initialized.")

Batches: 100%|██████████| 63/63 [00:04<00:00, 13.87it/s]

IncidentIQ initialized.


In [ ]:
QUERY = "node card failure"
TOP_K = 3

result = incidentiq.investigate(
    query=QUERY,
    top_k=TOP_K,
)

state = InvestigationState(
    query=QUERY,
    evidence=list(result.evidence),
    tool_calls=list(result.tool_calls),
    observations=list(result.analysis.observations),
    hypotheses=list(result.analysis.hypotheses),
    unknowns=list(result.analysis.unknowns),
    next_steps=list(result.analysis.next_steps),
    iteration=1,
)

state.evidence.extend(result.tool_evidence)

print("Initial investigation completed.")
print("Iteration:", state.iteration)
print("Evidence:", len(state.evidence))
print("Tool calls:", len(state.tool_calls))
print("Complete:", result.analysis.investigation_complete)

FUNCTION NAME: 'search_logs'
FUNCTION NAME TYPE: <class 'str'>
EXPECTED: 'search_logs'
EQUAL: True
Tool call : search_logs with arguments: {'query': 'R26-M0-N0-I:J18-U11', 'top_k': 10}
Tool returned 10 results.
Total tool evidence :  10
Tool evidence IDs:  [1795, 294, 1728, 1647, 1303, 1617, 1650, 1296, 1623, 1335]
Prompt: 0.000s | Gemini API: 10.037s | Parsing: 0.000s
Retrieval: 0.012s | Context: 0.001s | Patterns: 0.000s | Reasoning: 10.037s | Total: 10.050s
Initial investigation completed.
Iteration: 1
Evidence: 13
Tool calls: 1
Complete: True


In [4]:
result.analysis.investigation_complete

True

In [5]:
MAX_ITERATIONS = 3

while True:

    print(f"\n--- Investigation Iteration {state.iteration} ---")

    if result.analysis.investigation_complete:
        print("Agent reports investigation complete.")
        break

    if state.iteration >= MAX_ITERATIONS:
        print("Maximum investigation iterations reached.")
        break

    print("Agent requests another investigation iteration.")

    analyzer_result = incidentiq.analyzer.analyze(
        query=state.query,
        context={
            "evidence": state.evidence,
        },
        patterns=[],
        state=state,
    )

    result = analyzer_result

    state.observations = analyzer_result.analysis.observations
    state.hypotheses = analyzer_result.analysis.hypotheses
    state.unknowns = analyzer_result.analysis.unknowns
    state.next_steps = analyzer_result.analysis.next_steps

    state.tool_calls.extend(analyzer_result.tool_calls)
    state.evidence.extend(analyzer_result.tool_evidence)

    state.iteration += 1

print("\nInvestigation finished.")
print("Iterations:", state.iteration)
print("Evidence:", len(state.evidence))
print("Tool calls:", len(state.tool_calls))
print("Complete:", result.analysis.investigation_complete)


--- Investigation Iteration 1 ---
Agent reports investigation complete.

Investigation finished.
Iterations: 1
Evidence: 13
Tool calls: 1
Complete: True
